In [0]:

# Project base path
base_path = "/Volumes/adv_dev_westus3_demo_001/ecommerce_01/lakehouse_vol_01"

# Define source and target paths
raw_path = f"{base_path}/Raw_DS"
bronze_path = f"{base_path}/bronze"

# Display project paths
print("Raw data path:", raw_path)
print("Bronze data path:", bronze_path)

# Check that the folders are accessible
display(dbutils.fs.ls(base_path))


Raw data path: /Volumes/adv_dev_westus3_demo_001/ecommerce_01/lakehouse_vol_01/Raw_DS
Bronze data path: /Volumes/adv_dev_westus3_demo_001/ecommerce_01/lakehouse_vol_01/bronze


path,name,size,modificationTime
dbfs:/Volumes/adv_dev_westus3_demo_001/ecommerce_01/lakehouse_vol_01/Raw_DS/,Raw_DS/,0,1792667306000
dbfs:/Volumes/adv_dev_westus3_demo_001/ecommerce_01/lakehouse_vol_01/bronze/,bronze/,0,1792667307000
dbfs:/Volumes/adv_dev_westus3_demo_001/ecommerce_01/lakehouse_vol_01/gold/,gold/,0,1792667309000
dbfs:/Volumes/adv_dev_westus3_demo_001/ecommerce_01/lakehouse_vol_01/silver/,silver/,0,1792667308000


In [0]:
# Create a parameter widget
dbutils.widgets.text("file_name", "ALL", "Enter file name or ALL")

# Get the parameter value
file_name = dbutils.widgets.get("file_name").strip()

print("Selected parameter:", file_name)

Selected parameter: ALL


In [0]:

# Define the five required CSV files
required_files = [
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_customers_dataset.csv",
    "olist_products_dataset.csv",
    "olist_order_payments_dataset.csv"
]

# Select files using the parameter
if file_name.upper() == "ALL":
    selected_files = required_files
else:
    selected_files = [file_name]
    if not file_name.endswith(".csv"):
        selected_files = [file_name + ".csv"]

# Read each CSV file and save it as Parquet
for name in selected_files:
    if name not in required_files:
        raise ValueError(f"File is not in the required list: {name}")

    table_name = name.replace("olist_", "").replace("_dataset.csv", "")

    df = spark.read \
        .option("header", True) \
        .option("inferSchema", True) \
        .csv(f"{raw_path}/{name}")

    target_path = f"{bronze_path}/{table_name}"

    df.write \
        .mode("overwrite") \
        .parquet(target_path)

    print(f"Loaded {name} into {target_path}")
    print(f"Rows processed: {df.count()}")


Loaded olist_orders_dataset.csv into /Volumes/adv_dev_westus3_demo_001/ecommerce_01/lakehouse_vol_01/bronze/orders
Rows processed: 99441
Loaded olist_order_items_dataset.csv into /Volumes/adv_dev_westus3_demo_001/ecommerce_01/lakehouse_vol_01/bronze/order_items
Rows processed: 112650
Loaded olist_customers_dataset.csv into /Volumes/adv_dev_westus3_demo_001/ecommerce_01/lakehouse_vol_01/bronze/customers
Rows processed: 99441
Loaded olist_products_dataset.csv into /Volumes/adv_dev_westus3_demo_001/ecommerce_01/lakehouse_vol_01/bronze/products
Rows processed: 32951
Loaded olist_order_payments_dataset.csv into /Volumes/adv_dev_westus3_demo_001/ecommerce_01/lakehouse_vol_01/bronze/order_payments
Rows processed: 103886
